# Heizwärmebedarf im Geometrielauf des Gebäudesektormodells

Diese Fassung von [HD_Type_Analysis.ipynb](HD_Type_Analysis.ipynb)
wertet die vorhandenen Jahreswerte aus `building_sector_geo` aus. Sie zeigt,
wie sich der Heizwärmebedarf (`QH`) nach Grundgeometrie und Hüllvariante
verteilt und wie sich der Fensterflächenanteil im paarweisen Vergleich auswirkt.

Installiere im Repository einmalig `python -m pip install -e ".[analysis]"`
und wähle dieselbe Umgebung als Notebook-Kernel.

In [1]:
#| code-fold: true
#| code-summary: "Vorbereitung für lokale Ausführung"
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

from pexl import plot, reporting

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src" / "pexl").is_dir())
pio.renderers.default = "plotly_mimetype"

## Szenarionamen in Merkmale aufteilen

`EFH_1G_OFF_R000_F15|vor1919|Leicht` bezeichnet die Grundgeometrie
`EFH_1G`, den Bebauungsformcode `OFF`, die Rotation 0°, den
Fensterflächenanteil 15 %, die Hüllvariante `vor1919` und die Speichermasse
`Leicht`. Die Läufe verwenden für den Bestand die aufgezeichneten MFH-Hüllen
über alle Geometrieklassen hinweg. Die Hüllvariante für die Gruppierung
kommt deshalb aus dem Szenarionamen. Die aufgezeichnete Hülle bleibt zur
Kontrolle in der Tabelle erhalten.

In [2]:
PARQUET_PATH = ROOT / "data/building_typologies/building_sector_geo/scenarios.parquet"
columns = ["scenario_name", "preset_recorded_geometry", "preset_recorded_hulls",
           "QH", "PV_roof_share", "weather_name"]
scenarios = pd.read_parquet(PARQUET_PATH, columns=columns).set_index(
    "scenario_name", verify_integrity=True,
)
names = scenarios.index.to_series()
valid = names.str.fullmatch(r"[A-Za-z0-9_]+_[A-Z]+_R\d{3}_F\d{2}\|[^|]+\|[^|]+")
if not valid.all():
    raise ValueError(f"Unerwartetes Szenarionamensformat: {names[~valid].head().tolist()}")

parts = names.str.split("|", regex=False, expand=True)
parts.columns = ["Geometriepreset", "Hüllvariante", "Speichermasse"]
geometry = parts["Geometriepreset"].str.rsplit("_", n=3, expand=True)
geometry.columns = ["Grundgeometrie", "Bebauungsform", "Rotation", "Fensterflächenanteil"]
if not parts["Geometriepreset"].eq(scenarios["preset_recorded_geometry"]).all():
    raise ValueError("Geometrie im Szenarionamen und aufgezeichnetes Preset widersprechen sich.")

scenarios = scenarios.join(parts[["Hüllvariante", "Speichermasse"]]).join(geometry)
scenarios["Geometrieklasse"] = scenarios["Grundgeometrie"].str.split("_", n=1).str[0]
scenarios["Rotation"] = scenarios["Rotation"].str[1:].astype(int)
scenarios["Fensterflächenanteil"] = scenarios["Fensterflächenanteil"].str[1:].astype(int)
scenarios["Hüllgruppe"] = np.where(
    scenarios["Hüllvariante"].str.startswith("Neu_"), "Neubau", "Bestand",
)
scenarios["QH"] = pd.to_numeric(scenarios["QH"], errors="raise")
data = scenarios.loc[np.isfinite(scenarios["QH"])].copy()
if data.empty:
    raise ValueError("Der Export enthält keine gültigen QH-Werte.")

print(f"{len(data):,} auswertbare Szenarien von {len(scenarios):,}")
display(data[["Grundgeometrie", "Bebauungsform", "Rotation", "Fensterflächenanteil",
              "Hüllvariante", "Speichermasse", "QH"]].head())
display(data.groupby(["weather_name", "PV_roof_share"], dropna=False).size().rename("Szenarien"))
display(data.groupby(["Hüllvariante", "preset_recorded_hulls"]).size().rename("Szenarien"))

8,640 auswertbare Szenarien von 8,640


,Grundgeometrie,Bebauungsform,Rotation,Fensterflächenanteil,Hüllvariante,Speichermasse,QH
scenario_name,,,,,,,
EFH_1G_OFF_R000_F15|vor1919|Leicht,EFH_1G,OFF,0,15,vor1919,Leicht,481.726071
EFH_1G_OFF_R000_F15|vor1919|mittel,EFH_1G,OFF,0,15,vor1919,mittel,481.640990
EFH_1G_OFF_R000_F15|vor1919|schwer,EFH_1G,OFF,0,15,vor1919,schwer,481.727517
EFH_1G_OFF_R000_F15|1919-44|Leicht,EFH_1G,OFF,0,15,1919-44,Leicht,416.186632
EFH_1G_OFF_R000_F15|1919-44|mittel,EFH_1G,OFF,0,15,1919-44,mittel,415.600956


weather_name  PV_roof_share
Wien 2020     0.3              8640
Name: Szenarien, dtype: int64

Hüllvariante  preset_recorded_hulls
1919-44       MFH_1919 - 44            1080
1945-59       MFH_1945 - 59            1080
1960-79       MFH_1960 - 79            1080
1980-89       MFH_1980 - 89            1080
1990-99       MFH_1990 - 99            1080
Neu_20cm      WD2 (2/3, 20cm)          1080
Neu_30cm      WD3 (3/3, 30cm)          1080
vor1919       MFH_bis1919              1080
Name: Szenarien, dtype: int64

## Überblick nach Grundgeometrie und Hüllvariante

Die Boxplots zeigen die Streuung, die Heatmap zeigt die Mediane. Jede
Simulationsvariante zählt gleich. Die Ergebnisse beschreiben den untersuchten
Variantenraum. Für den Gebäudebestand wären eigene Gewichte erforderlich.

In [3]:
hull_order = ["vor1919", "1919-44", "1945-59", "1960-79",
              "1980-89", "1990-99", "Neu_20cm", "Neu_30cm"]
class_order = ["EFH", "RH", "MFH", "GWB", "NiWo"]
geometry_order = sorted(
    data["Grundgeometrie"].unique(),
    key=lambda name: (class_order.index(name.split("_")[0]), name),
)
labels = reporting.variables(data, "QH")["labels"]


box_colors = {"Bestand": ("#b9d7ea", "#36648b"),
              "Neubau": ("#c9e4c5", "#4b8a3f")}
qh_range = (0, 550)

for column, order in (("Grundgeometrie", geometry_order), ("Hüllvariante", hull_order)):
    groups = "Hüllgruppe" if column == "Hüllvariante" else None
    figure = plot.distribution(data, "QH", by=column, color=groups, labels=labels)
    plot.styles.style_boxes(figure, colors=box_colors)
    figure.update_yaxes(range=qh_range)
    figure.update_xaxes(categoryorder="array", categoryarray=order)
    figure.update_layout(title=f"Heizwärmebedarf nach {column}", boxmode="overlay")
    display(figure)

medians = data.pivot_table(index="Grundgeometrie", columns="Hüllvariante",
                           values="QH", aggfunc="median")
medians = medians.reindex(index=geometry_order, columns=hull_order)
display(px.imshow(medians, text_auto=".0f", aspect="auto", color_continuous_scale="Viridis",
                  title="Median des Heizwärmebedarfs",
                  labels={"color": labels["QH"]}))

## Hüllvarianten je Grundgeometrie und Geometrieklasse

`pexl.plot.distribution()` erzeugt ein Teilbild pro Wert in `facet`.
Die Teilbilder verwenden dieselbe QH-Skala. Je Grundgeometrie fließen
Bebauungsform, Rotation, Fensterflächenanteil und Speichermasse in die Streuung
ein. Auf Klassenebene werden zusätzlich die Grundgeometrien zusammengefasst.

Blau steht für Bestand, Grün für Neubau. Die rote Linie markiert den Median, das Dreieck den Mittelwert. Die Punkte markieren Ausreißer nach der Boxplot-Regel.

In [4]:
data["Grundgeometrie"] = pd.Categorical(data["Grundgeometrie"], geometry_order, ordered=True)
data["Geometrieklasse"] = pd.Categorical(data["Geometrieklasse"], class_order, ordered=True)

for facet, height in (("Grundgeometrie", 1120), ("Geometrieklasse", 690)):
    figure = plot.distribution(
        data.sort_values(facet), "QH", by="Hüllvariante", color="Hüllgruppe",
        facet=facet, facet_wrap=3, labels=labels,
    )
    plot.styles.style_boxes(figure, colors=box_colors)
    figure.update_yaxes(range=qh_range)
    counts = data[facet].value_counts()
    for heading in figure.layout.annotations:
        value = heading.text.split("=", 1)[1]
        heading.text = f"{value} (n={counts[value]:,})"
    figure.update_xaxes(categoryorder="array", categoryarray=hull_order, tickangle=-35)
    figure.update_layout(title=f"Heizwärmebedarf nach Hüllvariante je {facet}", height=height,
                         boxmode="overlay")
    display(figure)

## Fensterflächenanteil bei sonst gleichen Varianten

Die Differenz ist QH bei 40 % minus QH bei 15 % Fensterflächenanteil.
Innerhalb jedes Paars bleiben Grundgeometrie, Bebauungsform, Rotation,
Hüllvariante und Speichermasse gleich. Gezeigt werden nur vollständige Paare.
Positive Werte stehen für einen höheren Heizwärmebedarf bei 40 %.

In [5]:
pair_keys = ["Geometrieklasse", "Grundgeometrie", "Bebauungsform", "Rotation",
             "Hüllvariante", "Speichermasse", "weather_name", "PV_roof_share"]
paired = data.pivot(index=pair_keys, columns="Fensterflächenanteil", values="QH")
paired = paired.dropna(subset=[15, 40]).reset_index()
paired["QH_Differenz"] = paired[40] - paired[15]
summary = paired.groupby("Geometrieklasse", observed=True)["QH_Differenz"].agg(
    Paare="size", Median="median", Minimum="min", Maximum="max",
)
display(summary.round(1))

figure = plot.distribution(
    paired.sort_values("Geometrieklasse"), "QH_Differenz",
    by="Hüllvariante", facet="Geometrieklasse", facet_wrap=3,
    labels={"QH_Differenz": f"Δ {labels['QH']}"},
)
plot.styles.style_boxes(figure, colors=box_colors)
figure.add_hline(y=0, line_dash="dash", line_color="gray")
figure.update_xaxes(categoryorder="array", categoryarray=hull_order, tickangle=-35)
figure.update_layout(title="Änderung des Heizwärmebedarfs von 15 % auf 40 % Fensterfläche",
                     height=690)
display(figure)

,Paare,Median,Minimum,Maximum
Geometrieklasse,,,,
EFH,576,2.7,-46.5,43.5
RH,288,2.3,-35.4,31.9
MFH,576,1.2,-28.9,25.4
GWB,576,1.4,-20.6,18.8
NiWo,864,0.6,-14.0,12.0


## Vergleich mit den Energieausweis-Referenzwerten aus dem Ausgangsnotebook

Die Referenztabelle und der Faktor 1,25 stammen aus dem Ausgangsnotebook.
Ausgewertet wird der Quotient aus angepasstem Referenzwert und
Simulationsmedian. Für `1960-79` sind zwei Referenzepochen dargestellt.
Für RH werden dieselben EAW-Referenzwerte wie für EFH angesetzt. Für Neubau
liegt keine Referenz vor. Die ähnlichen,
teilweise versetzten Epochen und die im Geometrielauf gemeinsam verwendeten
MFH-Hüllen sind bei der Interpretation mitzulesen. Die Quotienten dienen der
Gegenüberstellung in diesem Datensatz.

Oben stehen die Energieausweiswerte als Rauten über den Boxplots. Unten steht ihr Verhältnis zum jeweiligen Simulationsmedian. Orange markiert die zweite Referenzepoche 1971 bis 1980.

Die Verhältnisachse zeigt den Bereich von 0 bis 2. Die Tabelle enthält die berechneten Werte.

In [6]:
referenz = pd.DataFrame({
    "Epoche": [
        "Vor 1919",
        "1919 bis 1944",
        "1945 bis 1960",
        "1961 bis 1970",
        "1971 bis 1980",
        "1981 bis 1990",
        "1991 bis 2000",
        "2001 bis 2010",
    ],
    "EFH": [
        228.9, 285.3, 296.6, 199.7,
        175.7, 156.3, 76.6, 80.3,
    ],
    "MFH": [
        162.2, 169.1, 157.9, 142.9,
        127.4, 113.5, 57.5, 60.0,
    ],
    "GWB": [
        123.3, 126.9, 117.8, 108.2,
        97.2, 86.8, 46.5, 47.9,
    ],
    "NiWo": [
        151.3, 153.9, 140.8, 128.4,
        112.5, 100.4, 53.7, 55.2,
    ],
}).set_index("Epoche")

referenz["RH"] = referenz["EFH"]
eaw_sources = {"RH": "EFH", "NiWo": "DLG"}
referenz_faktor = 1.25
epoch_mapping = {
    "vor1919": ["Vor 1919"],
    "1919-44": ["1919 bis 1944"],
    "1945-59": ["1945 bis 1960"],
    "1960-79": ["1961 bis 1970", "1971 bis 1980"],
    "1980-89": ["1981 bis 1990"],
    "1990-99": ["1991 bis 2000"],
}
epochs = pd.Series(epoch_mapping, name="Epoche").rename_axis("Hüllvariante").explode().reset_index()
reference = (referenz.mul(referenz_faktor).rename_axis("Geometrieklasse", axis=1)
             .stack().rename("QH_Referenz").reset_index())
simulation = (
    data.groupby(["Geometrieklasse", "Hüllvariante"], observed=True)["QH"]
    .agg(Szenarien="size", QH_Median="median").reset_index()
)
comparison = simulation.merge(epochs, on="Hüllvariante").merge(
    reference, on=["Geometrieklasse", "Epoche"],
)
comparison["Referenz_zu_Median"] = comparison["QH_Referenz"].div(
    comparison["QH_Median"].where(comparison["QH_Median"].ne(0)),
)
display(comparison.round(2))

reference_classes = [name for name in class_order if name in comparison["Geometrieklasse"].values]
figure = make_subplots(
    rows=2, cols=len(reference_classes), shared_xaxes=True, shared_yaxes="rows",
    row_heights=[0.6, 0.4], horizontal_spacing=0.055, vertical_spacing=0.12,
    subplot_titles=[f"{name} · EAW {eaw_sources.get(name, name)}"
                    for name in reference_classes] + [""] * len(reference_classes),
)
positions = {name: i for i, name in enumerate(hull_order)}
for column, name in enumerate(reference_classes, start=1):
    subset = data.loc[data["Geometrieklasse"] == name]
    boxes = plot.distribution(subset, "QH", by="Hüllvariante", color="Hüllgruppe", labels=labels)
    for trace in boxes.data:
        trace.x = [positions[hull] for hull in trace.x]
        trace.showlegend = column == 1 and trace.type == "box"
        figure.add_trace(trace, row=1, col=column)

    points = comparison.loc[comparison["Geometrieklasse"] == name]
    for second, color, label in (
        (False, "#171717", "Energieausweis"),
        (True, "#d97706", "Zweite Epoche: 1971–1980"),
    ):
        other_epoch = points["Epoche"].eq("1971 bis 1980")
        entries = points.loc[other_epoch.eq(second)]
        x = entries["Hüllvariante"].map(positions)
        offset = np.where(entries["Hüllvariante"].eq("1960-79"), 0.16 if second else -0.16, 0)
        for row, variable in ((1, "QH_Referenz"), (2, "Referenz_zu_Median")):
            figure.add_trace(go.Scatter(
                x=x + offset, y=entries[variable], text=entries["Epoche"],
                mode="markers", marker=dict(symbol="diamond", size=8, color=color,
                                            line=dict(color="white", width=0.6)),
                name=label, legendgroup=label, showlegend=row == 1 and column == 1,
                hovertemplate="%{text}: %{y:.2f}<extra></extra>",
            ), row=row, col=column)

figure.update_xaxes(type="linear", range=(-0.5, len(hull_order) - 0.5),
                    tickmode="array", tickvals=list(positions.values()),
                    ticktext=[f"{name}*" if name == "1960-79" else name for name in hull_order],
                    tickangle=-45)
figure.update_xaxes(showticklabels=False, row=1)
figure.update_yaxes(range=qh_range, row=1)
figure.update_yaxes(title_text=labels["QH"], row=1, col=1)
plot.styles.style_boxes(figure, colors=box_colors)
figure.update_yaxes(range=(0, 2), row=2)
figure.update_yaxes(title_text="Energieausweis / Median", row=2, col=1)
figure.add_hline(y=1, line_dash="dash", line_color="gray", row=2)
figure.update_layout(
    template="plotly_white", boxmode="overlay", height=660,
    title="Heizwärmebedarf und Energieausweis-Referenz je Geometrieklasse",
    legend=dict(orientation="h", y=-0.28, x=0), margin=dict(b=115),
)
display(figure)

,Geometrieklasse,Hüllvariante,Szenarien,QH_Median,Epoche,QH_Referenz,Referenz_zu_Median
0,EFH,1919-44,216,339.02,1919 bis 1944,356.62,1.05
1,EFH,1945-59,216,345.45,1945 bis 1960,370.75,1.07
2,EFH,1960-79,216,342.52,1961 bis 1970,249.62,0.73
3,EFH,1960-79,216,342.52,1971 bis 1980,219.62,0.64
4,EFH,1980-89,216,197.08,1981 bis 1990,195.38,0.99
5,EFH,1990-99,216,157.87,1991 bis 2000,95.75,0.61
6,EFH,vor1919,216,392.21,Vor 1919,286.12,0.73
7,RH,1919-44,108,258.89,1919 bis 1944,356.62,1.38
8,RH,1945-59,108,264.21,1945 bis 1960,370.75,1.40
9,RH,1960-79,108,256.22,1961 bis 1970,249.62,0.97


## EAW-Vergleich je Grundgeometrietyp

Je Hüllvariante zeigt die Violine die QH-Verteilung. Ihre Breite skaliert
mit der Anzahl der Szenarien. In diesem Lauf sind es 108 pro Hüllvariante
und Grundgeometrietyp. Jeder Punkt zeigt eine Simulation und ist horizontal
leicht versetzt. Die Farbe steht für den Fensterflächenanteil, die
Symbolform für die Bebauungsform. Orientierung und Speichermasse stehen
zusammen mit dem Szenarionamen im Hover.

Die Rauten zeigen den EAW-Referenzwert der jeweiligen Geometrieklasse
einschließlich Faktor 1,25. Grundgeometrietypen derselben Klasse teilen
sich die Referenz. RH übernimmt die EFH-Werte. Für Neubau sind nur die
Simulationen dargestellt.

In [7]:
colors = {15: "#0072b2", 25: "#cc79a7", 40: "#009e73"}
symbols = {"OFF": "circle", "GEK": "square", "GES": "triangle-up"}
eaw = epochs.merge(reference, on="Epoche")
second_epoch = "1971 bis 1980"


def eaw_violin_figure(*, split_by_form=False):
    n_columns = len(symbols) if split_by_form else 2
    panels = (
        [(kind, form) for kind in geometry_order for form in symbols]
        if split_by_form else [(kind, None) for kind in geometry_order]
    )
    n_rows = (len(panels) + n_columns - 1) // n_columns
    panel_titles = [f"{kind} · {form}" if form else kind for kind, form in panels]
    plot_height = 315 * n_rows
    height = plot_height + 215
    figure = make_subplots(
        rows=n_rows, cols=n_columns, shared_yaxes="rows",
        subplot_titles=panel_titles, vertical_spacing=80 / plot_height,
        horizontal_spacing=0.07,
    )
    rng = np.random.default_rng(42)

    for panel, (kind, form) in enumerate(panels):
        row, col = panel // n_columns + 1, panel % n_columns + 1
        subset = data.loc[data["Grundgeometrie"].eq(kind)]
        if form is not None:
            subset = subset.loc[subset["Bebauungsform"].eq(form)]
        if subset.empty:
            continue
        for hull, group in subset.groupby("Hüllvariante", sort=False):
            fill, outline = box_colors[group["Hüllgruppe"].iloc[0]]
            figure.add_trace(go.Violin(
                x=[positions[hull]] * len(group), y=group["QH"], name=hull,
                points=False, scalemode="count", scalegroup="QH", spanmode="hard",
                fillcolor=fill, line=dict(color=outline, width=1.1), showlegend=False,
            ), row=row, col=col)

        for share, variants in subset.groupby("Fensterflächenanteil", sort=True):
            jitter = rng.uniform(-0.15, 0.15, len(variants))
            figure.add_trace(go.Scatter(
                x=variants["Hüllvariante"].map(positions) + jitter,
                y=variants["QH"], mode="markers", text=variants.index,
                customdata=variants[["Bebauungsform", "Rotation", "Speichermasse"]],
                name=f"{share} %", legendgroup=f"ff-{share}",
                legendgrouptitle_text="Fensterflächenanteil", showlegend=panel == 0,
                marker=dict(size=4.5, color=colors[share], opacity=0.66,
                            symbol=variants["Bebauungsform"].map(symbols)),
                hovertemplate=(
                    f"%{{text}}<br>QH: %{{y:.1f}}<br>Fensterflächenanteil: {share} %"
                    "<br>Bebauungsform: %{customdata[0]}<br>Orientierung: %{customdata[1]}°"
                    "<br>Speichermasse: %{customdata[2]}<extra></extra>"
                ),
            ), row=row, col=col)

        geometry_class = subset["Geometrieklasse"].iloc[0]
        refs = eaw.loc[eaw["Geometrieklasse"].eq(geometry_class)]
        for second, color, label in (
            (False, "#171717", "EAW-Referenz"),
            (True, "#d97706", "Zweite EAW-Epoche"),
        ):
            entries = refs.loc[refs["Epoche"].eq(second_epoch).eq(second)]
            offset = np.where(entries["Hüllvariante"].eq("1960-79"), 0.16 if second else -0.16, 0)
            figure.add_trace(go.Scatter(
                x=entries["Hüllvariante"].map(positions) + offset,
                y=entries["QH_Referenz"], text=entries["Epoche"], mode="markers",
                marker=dict(symbol="diamond", size=9, color=color,
                            line=dict(color="white", width=0.8)),
                name=label, legendgroup=label, showlegend=panel == 0,
                hovertemplate=(f"EAW {eaw_sources.get(geometry_class, geometry_class)}"
                               " · %{text}: %{y:.1f}<extra></extra>"),
            ), row=row, col=col)

    for form, symbol in symbols.items():
        figure.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers", legend="legend2", name=form,
            marker=dict(symbol=symbol, color="#555555", size=7),
        ), row=1, col=1)

    figure.update_xaxes(
        type="linear", range=(-0.5, len(hull_order) - 0.5),
        tickmode="array", tickvals=list(positions.values()), ticktext=hull_order, tickangle=-30,
    )
    figure.update_yaxes(rangemode="tozero", gridcolor="#e6e6e6")
    for row in range(1, n_rows + 1):
        figure.update_yaxes(title_text=labels["QH"], row=row, col=1)

    title = "Heizwärmebedarf und Energieausweis-Referenz je Grundgeometrietyp"
    if split_by_form:
        title += " und Bebauungsform"
    figure.update_layout(
        title=dict(text=title, y=1 - 25 / height, yanchor="top"),
        template="plotly_white", violinmode="overlay", height=height,
        margin=dict(t=150, b=65),
        legend=dict(orientation="h", x=0, y=1 + 75 / plot_height, itemsizing="constant"),
        legend2=dict(title_text="Bebauungsform", orientation="h", x=0, y=1 + 38 / plot_height,
                     itemclick=False, itemdoubleclick=False),
    )
    return figure


display(eaw_violin_figure())

### EAW-Vergleich getrennt nach Bebauungsform

Pro Grundgeometrietyp stehen OFF, GEK und GES nebeneinander. Je
Hüllvariante und Bebauungsform sind 36 Simulationen dargestellt.
Die drei Teilbilder einer Zeile teilen sich eine QH-Skala. Jede Zeile skaliert separat. Die Farben zeigen weiterhin
den Fensterflächenanteil. Die EAW-Klassenreferenz gilt für alle drei
Spalten eines Grundgeometrietyps.

In [8]:
display(eaw_violin_figure(split_by_form=True))